# 03 — Road Network Analysis

Load Ministry of Transport road network. Filter to autopistas (AP-), autovías (A-), carreteras nacionales (N-). Remove urban sections. Build the interurban road graph/geometry.

## Data Inputs
- `data/processed/roads_clean.geojson`
- `data/raw/additional/tent_corridors/` — TEN-T corridor maps
- `data/raw/additional/dgt_imd_traffic/` — IMD traffic counts

## Data Outputs
- `data/processed/interurban_roads.geojson` — filtered interurban road network
- `data/processed/road_segments_with_imd.csv` — segments with traffic counts

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from src.constants import MAX_STATION_SPACING_KM, AFIR_SPACING_KM

PROCESSED = Path('data/processed')

# Load cleaned roads
roads = gpd.read_file(PROCESSED / 'roads_clean.geojson')
print(f"Total road segments: {roads.shape[0]}")
print(f"Road types:\n{roads['Tipo_de_via'].value_counts()}")

## Filter to Interurban Roads

Per datathon rules: only autopistas (AP-), autovías (A-), and carreteras nacionales (N-) as classified by Ministry of Transport. Urban road sections are excluded.

In [ ]:
# Filter to AP-, A-, N- roads only
interurban = roads[
    roads['Carretera'].str.match(r'^(AP-|A-|N-)', na=False)
].copy()
print(f"Interurban segments: {interurban.shape[0]} (from {roads.shape[0]})")
print(f"\nBy prefix:")
print(interurban['road_prefix'].value_counts())
print(f"\nBy road type:")
print(interurban['Tipo_de_via'].value_counts())
print(f"\nTotal network length: {interurban['length_km'].sum():,.0f} km")

# Exclude urban sections using Tipo_de_via:
# AP- and A- roads are grade-separated highways (inherently interurban)
# N- roads with 'Carretera convencional' may pass through cities,
# but we keep them as they are classified as interurban by the Ministry
# Urban filtering will be refined when we place stations (avoid city cores)

# Add a segment_id for downstream joins
interurban['segment_id'] = range(len(interurban))

interurban.head()

## Associate IMD Traffic with Road Segments

Match DGT traffic counting stations to road segments using road name (`Carretera`) and kilometric point (`PK`). Aggregate mean IMD per segment.

In [ ]:
# Load IMD traffic stations
imd = gpd.read_file(PROCESSED / 'imd_traffic_clean.geojson')

# Check what IMD columns we have
imd_cols = [c for c in imd.columns if 'IMD' in c or 'imd' in c.lower()]
print(f"IMD columns available: {imd_cols}")
print(f"IMD stations: {imd.shape[0]}")

# Normalize road names for matching (strip whitespace)
imd['Carretera_clean'] = imd['Carretera'].str.strip()
interurban['Carretera_clean'] = interurban['Carretera'].str.strip()

# Match: for each IMD station, find the road segment where
# station.Carretera == segment.Carretera AND segment.PK_inicio <= station.PK <= segment.PK_fin
imd_match = imd[['Carretera_clean', 'PK', 'IMD_total', 'IMD_ligeros', 'IMD_pesados']].copy()
imd_match = imd_match.dropna(subset=['IMD_total'])

# Convert PK columns to numeric
for col in ['PK', 'IMD_total', 'IMD_ligeros', 'IMD_pesados']:
    imd_match[col] = pd.to_numeric(imd_match[col], errors='coerce')

segments_with_pk = interurban[['segment_id', 'Carretera_clean', 'PK_inicio', 'PK_fin']].copy()
for col in ['PK_inicio', 'PK_fin']:
    segments_with_pk[col] = pd.to_numeric(segments_with_pk[col], errors='coerce')

# Merge on road name then filter by PK range
merged = imd_match.merge(segments_with_pk, on='Carretera_clean')
matched = merged[
    (merged['PK'] >= merged['PK_inicio']) &
    (merged['PK'] <= merged['PK_fin'])
]

# Aggregate per segment
seg_imd = matched.groupby('segment_id').agg(
    imd_total=('IMD_total', 'mean'),
    imd_ligeros=('IMD_ligeros', 'mean'),
    imd_pesados=('IMD_pesados', 'mean'),
    n_stations=('IMD_total', 'count'),
).reset_index()

print(f"\nSegments with IMD data: {len(seg_imd)} / {len(interurban)}")
print(f"IMD total range: {seg_imd.imd_total.min():.0f} – {seg_imd.imd_total.max():.0f}")
print(f"Mean IMD: {seg_imd.imd_total.mean():.0f}")

# Join back to interurban
interurban = interurban.merge(seg_imd, on='segment_id', how='left')

# For segments without IMD, fill with road-level median
road_medians = interurban.groupby('Carretera_clean')['imd_total'].transform('median')
interurban['imd_total'] = interurban['imd_total'].fillna(road_medians)
# Still missing? Use global median
global_median = interurban['imd_total'].median()
interurban['imd_total'] = interurban['imd_total'].fillna(global_median)

print(f"\nAfter filling: {interurban['imd_total'].notna().sum()} / {len(interurban)} segments have IMD")

## Tag TEN-T Corridor Segments

Segments on TEN-T core network require 60 km charger spacing per AFIR regulation (vs 150 km for other roads).

In [ ]:
# Tag TEN-T corridors
# Use the TENT field from the original road data if available
if 'TENT' in interurban.columns:
    interurban['is_tent'] = interurban['TENT'].notna() & (interurban['TENT'] != '')
    print(f"TEN-T segments (from road data): {interurban['is_tent'].sum()}")
else:
    # Fall back to spatial join with TEN-T corridor GeoJSON
    tent = gpd.read_file(PROCESSED / 'tent_corridors.geojson')
    tent_roads = tent['Carretera'].dropna().unique() if 'Carretera' in tent.columns else []
    interurban['is_tent'] = interurban['Carretera'].isin(tent_roads)
    print(f"TEN-T segments (from corridor file): {interurban['is_tent'].sum()}")

# Set spacing constraint
interurban['max_spacing_km'] = np.where(
    interurban['is_tent'], AFIR_SPACING_KM, MAX_STATION_SPACING_KM
)
print(f"\nSpacing: TEN-T = {AFIR_SPACING_KM} km, Other = {MAX_STATION_SPACING_KM} km")
print(f"TEN-T road names: {interurban[interurban['is_tent']]['Carretera'].unique()[:20]}")

# Spatial join with provinces for geographic context
provinces = gpd.read_file(PROCESSED / 'provinces.geojson')
# Use centroid of each road segment for province assignment
centroids = interurban.copy()
centroids['geometry'] = centroids.geometry.centroid
centroids = centroids.sjoin(provinces[['geometry', 'cod_prov']], how='left', predicate='within')
interurban['cod_prov'] = centroids['cod_prov'].values

print(f"\nSegments by province: {interurban['cod_prov'].nunique()} provinces covered")

## Summary & Save Outputs

In [ ]:
# Summary statistics
print("=" * 60)
print("INTERURBAN ROAD NETWORK ANALYSIS — SUMMARY")
print("=" * 60)
print(f"Total interurban segments: {len(interurban)}")
print(f"Total network length: {interurban['length_km'].sum():,.0f} km")
print(f"TEN-T segments: {interurban['is_tent'].sum()} ({interurban['is_tent'].mean()*100:.1f}%)")
print(f"Provinces covered: {interurban['cod_prov'].nunique()}")
print(f"\nIMD traffic stats:")
print(interurban['imd_total'].describe().round(0))
print(f"\nBy road prefix:")
print(interurban.groupby('road_prefix').agg(
    segments=('segment_id', 'count'),
    total_km=('length_km', 'sum'),
    mean_imd=('imd_total', 'mean'),
).round(0))

# Save interurban roads GeoJSON
out_roads = PROCESSED / 'interurban_roads.geojson'
interurban.to_file(out_roads, driver='GeoJSON')
print(f"\nSaved: {out_roads} ({len(interurban)} segments)")

# Save road segments with IMD as CSV (flat, no geometry)
cols_csv = ['segment_id', 'Carretera', 'road_prefix', 'PK_inicio', 'PK_fin',
            'length_km', 'imd_total', 'imd_ligeros', 'imd_pesados',
            'n_stations', 'is_tent', 'max_spacing_km', 'cod_prov']
out_csv = PROCESSED / 'road_segments_with_imd.csv'
interurban[cols_csv].to_csv(out_csv, index=False)
print(f"Saved: {out_csv} ({len(interurban)} rows)")